In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import umap
import pickle
import sklearn.decomposition

## 1. t-SNE of segmented XRD

In [ ]:
tsne_diccio = pickle.load(open('./tsne_diccio_300_ee.pkl','rb'))
tsne_diccio[12] = pickle.load(open('./tsne_diccio_orxrd.pkl','rb'))[300]

In [ ]:
dbscan_diccio = dict()
for k in sorted(tsne_diccio.keys()):
    
    plt.figure()
    print(k)
    plt.scatter(tsne_diccio[k].embedding_[:,0],
                tsne_diccio[k].embedding_[:,1], s=0.1)
    plt.show()
    
    dbscan_diccio[k] = dict()
    for min_size in [5,10,15,20,25,30,35,40]:
        dbscan = sklearn.cluster.HDBSCAN(min_cluster_size=min_size)#, cluster_selection_epsilon=1)
        dbscan = dbscan.fit(tsne_diccio[k].embedding_)

        dbscan_diccio[k][min_size] = dbscan

        plt.figure()
        print(len(np.unique(dbscan.labels_)))
        for label in np.unique(dbscan.labels_):

            idxs = np.argwhere(dbscan.labels_ == label)[:,0]

            if label == -1:
                plt.scatter(tsne_diccio[k].embedding_[idxs,0], 
                            tsne_diccio[k].embedding_[idxs,1], s=1, color='black', alpha=0.1)
            else:
                plt.scatter(tsne_diccio[k].embedding_[idxs,0], 
                            tsne_diccio[k].embedding_[idxs,1], s=1, alpha=0.5)
                plt.annotate(text=f"{label}", xy = tsne_diccio[k].embedding_[idxs].mean(axis=0), fontsize=9)
        plt.show()

In [ ]:
with open('hdbscan_300_ee.pkl','wb') as f:
    pickle.dump(dbscan_diccio, f)

In [ ]:
clustered_samples = list()
for k in dbscan_diccio.keys():
    for sm in dbscan_diccio[k].keys():
        clustered_samples += [[k, sm, 
                               100*(dbscan_diccio[k][sm].labels_ != -1).sum()/(dbscan_diccio[k][sm].labels_.shape[0]),
                               len(np.unique(dbscan_diccio[k][sm].labels_))]]
        
df_clustered = pd.DataFrame(clustered_samples)
df_clustered.sort_values(by=2, ascending=False)